# Part 2 - CTI Alert Generation and LLM-Based Enrichment
### LLM-Assisted Cyber Threat Intelligence for 5G O-RAN Security

This notebook covers **Part 2** of the exercise:
1. Load Part 1's per-event predictions
2. Build a structured CTI alert per event
3. Enrich each alert using a locally hosted LLM via **Ollama**
4. Run an automated hallucination/consistency check on the LLM output
5. Export enriched alerts + one formatted sample incident report

**LLM used:** a small, locally hosted open-weight model via Ollama (default: `llama3.2:3b`).
The exercise explicitly favours lightweight local models over large commercial ones, and a
3B model is fast enough to run this loop interactively on a laptop CPU.

> **Before running:** make sure Ollama is running (`ollama serve`, or it auto-starts as a
> service after install) and the model is pulled: `ollama pull llama3.2:3b`.
> Set `USE_MOCK = False` below once you've confirmed Ollama responds (see the connectivity
> check in Section 1).


In [4]:
import os
import json
import re
import time
import requests
import warnings
warnings.filterwarnings("ignore")

PROCESSED_DIR = "../data/processed"
OUTPUTS_DIR = "../outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.2:3b"  

# Set to False once Ollama connectivity is confirmed in Section 1.
USE_MOCK = False


## 1. Check Ollama connectivity

Run this first. If it prints a successful response, set `USE_MOCK = False` in the cell above
and re-run from the top.


In [5]:
def ollama_healthcheck():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        print("Ollama is reachable. Installed models:", models)
        if OLLAMA_MODEL not in models and not any(OLLAMA_MODEL.split(":")[0] in m for m in models):
            print(f"WARNING: '{OLLAMA_MODEL}' not found in installed models. "
                  f"Run: ollama pull {OLLAMA_MODEL}")
        return True
    except Exception as e:
        print("Could not reach Ollama:", e)
        print("Make sure Ollama is installed and running (`ollama serve`), "
              "or leave USE_MOCK = True to test the pipeline logic without it.")
        return False

ollama_healthcheck()


Ollama is reachable. Installed models: ['llama3.2:3b', 'llama3.2:latest']


True

## 2. Load Part 1 outputs

Uses `data/processed/test_predictions.json` produced at the end of Part 1's notebook.


In [6]:
with open(os.path.join(PROCESSED_DIR, "test_predictions.json")) as f:
    all_records = json.load(f)

print(f"Loaded {len(all_records)} test-event records.")
print("Classes present:", sorted(set(r["predicted_class"] for r in all_records)))


Loaded 344764 test-event records.
Classes present: ['benign', 'bruteforce', 'ddos', 'dos', 'probe', 'web']


## 3. Threat knowledge base

A short, factual, category-level description of expected behaviour for each class. This is
what the exercise calls "relevant threat knowledge" -- it grounds the LLM in general,
textbook-level attack behaviour without being tied to this specific testbed, so the LLM has
something concrete to correlate the alert against rather than reasoning from the class name
alone.


In [7]:
THREAT_KNOWLEDGE = {
    "benign": "Normal operational traffic. No anomalous behaviour expected; connections "
              "typically complete cleanly (SF state) with proportionate byte/packet counts "
              "in both directions.",
    "dos": "Denial of Service: a single-source attempt to exhaust a target's resources or "
           "connection capacity. Often shows as many short-lived or incomplete connections "
           "(e.g. S0/REJ states), small packet sizes, and high frequency toward one destination.",
    "ddos": "Distributed Denial of Service: same resource-exhaustion goal as DoS but from "
            "many sources simultaneously, typically producing a higher-volume, higher-"
            "frequency burst of low-duration flows, sometimes spanning multiple protocols "
            "(TCP/UDP/ICMP).",
    "probe": "Reconnaissance/scanning: an attacker enumerating live hosts, open ports, or "
             "running services ahead of a more targeted attack. Shows as many short "
             "connections to varied or sequential ports with little or no data transferred "
             "(REJ/S0 states).",
    "bruteforce": "Repeated authentication attempts against a service (commonly SSH or FTP) "
                  "to guess valid credentials. Shows as multiple connection attempts to the "
                  "same service/port from a small number of sources within a short window, "
                  "often with completed handshakes (SF) but small, repetitive payloads.",
    "web": "Web application attack (e.g. SQL injection, XSS): exploitation attempts embedded "
           "in HTTP requests targeting application input handling. Shows as HTTP flows with "
           "unusual request methods, parameters, or payload patterns.",
}

FEATURE_LABELS = {
    "proto": "transport protocol", "service": "application service",
    "duration": "flow duration (s)", "src_bytes": "bytes sent by source",
    "dst_bytes": "bytes sent by destination", "conn_state": "connection state",
    "history": "TCP flag history", "src_pkts": "packets from source",
    "dst_pkts": "packets from destination", "src_port": "source port",
    "dst_port": "destination port",
}


## 4. Build the structured CTI alert

Required fields per the brief: predicted class, confidence, alternative predictions, network
observations, affected component, detection timestamp. **The true label is never included.**


In [8]:
def infer_affected_component(record: dict) -> str:
    """Non-invented affected-component label based on destination port/service. Defaults to
    the O-Cloud Edge Server, the documented target of this testbed\'s attack traffic, rather
    than fabricating a specific network element the data doesn\'t actually identify."""
    feats = record.get("raw_features", {})
    dst_port = feats.get("dst_port")
    service = feats.get("service", "-")
    if dst_port == 22 or service == "ssh":
        return "O-Cloud Edge Server -- SSH management interface"
    if dst_port in (80, 443) or service in ("http", "ssl"):
        return "O-Cloud Edge Server -- HTTP/HTTPS service"
    return "O-Cloud Edge Server (O-RAN O-Cloud)"


def build_cti_alert(record: dict) -> dict:
    feats = record.get("raw_features", {})
    observations = {FEATURE_LABELS.get(k, k): v for k, v in feats.items()}
    return {
        "alert_id": record["event_id"],
        "detection_timestamp": record.get("timestamp", ""),
        "predicted_threat_class": record["predicted_class"],
        "prediction_confidence": record["confidence"],
        "alternative_predictions": record.get("alternative_predictions", []),
        "affected_network_component": infer_affected_component(record),
        "network_observations": observations,
    }


# sanity check: confirm no ground-truth leakage into the alert
sample_alert = build_cti_alert(all_records[0])
alert_str = json.dumps(sample_alert)
assert "true_label" not in alert_str and "true_attack_type" not in alert_str, "LEAK DETECTED"
print("Leakage check passed -- no ground truth in the alert.")
print(json.dumps(sample_alert, indent=2))


Leakage check passed -- no ground truth in the alert.
{
  "alert_id": "CamG7QmkphbjT5xn2",
  "detection_timestamp": "",
  "predicted_threat_class": "web",
  "prediction_confidence": 1.0,
  "alternative_predictions": [
    {
      "class": "web",
      "confidence": 1.0
    },
    {
      "class": "benign",
      "confidence": 0.0
    },
    {
      "class": "dos",
      "confidence": 0.0
    }
  ],
  "affected_network_component": "O-Cloud Edge Server -- HTTP/HTTPS service",
  "network_observations": {
    "source port": 60368,
    "destination port": 80,
    "transport protocol": "tcp",
    "application service": "http",
    "flow duration (s)": 0.14495,
    "bytes sent by source": 142.0,
    "bytes sent by destination": 686.0,
    "connection state": "SF",
    "missed_bytes": 0,
    "TCP flag history": "ShADadfF",
    "packets from source": 6,
    "src_ip_bytes": 462,
    "packets from destination": 5,
    "dst_ip_bytes": 954,
    "ip_proto": 6,
    "http_trans_depth": 1,
    "files_t

## 5. Prompt template

Explicitly instructs the model not to invent indicators, to reflect uncertainty when
confidence is not high, and to return a single parseable JSON object.


In [9]:
def build_prompt(alert: dict, knowledge: str) -> str:
    return f"""You are a cyber threat intelligence assistant supporting a 5G O-RAN security analyst.

You are given a structured alert produced by a machine-learning intrusion detector, and a short
reference description of the predicted threat category. You do NOT have access to the ground
truth label -- only the alert below.

STRUCTURED ALERT:
{json.dumps(alert, indent=2)}

REFERENCE THREAT KNOWLEDGE for "{alert['predicted_threat_class']}":
{knowledge}

Using ONLY the information above, respond with a single JSON object with exactly these fields:
- "contextualisation": one or two sentences explaining what this alert means in plain terms
- "correlation_with_known_behaviour": how the network_observations do or do not align with the
  reference threat knowledge given above
- "severity": one of "Low", "Medium", "High", "Critical"
- "possible_impact": short description of what could happen if this is a real, successful attack
- "immediate_response": a concrete short-term action a security analyst could take right now
- "long_term_mitigation": a concrete longer-term mitigation
- "human_review_required": true or false
- "human_review_reason": one sentence justifying the human_review_required value, referencing
  the prediction_confidence and/or alternative_predictions

IMPORTANT CONSTRAINTS:
- Do not invent IP addresses, hostnames, usernames, file names, or any indicator not present in
  the alert above.
- If the evidence is ambiguous or the confidence is not high, say so explicitly rather than
  asserting certainty.
- Base "severity" and "human_review_required" on the actual prediction_confidence and
  alternative_predictions given, not on the label name alone.

Respond with ONLY the JSON object, no other text."""


## 6. LLM call (Ollama) + mock fallback

`call_ollama` hits the real local model. `mock_ollama_call` is a stand-in used only while
`USE_MOCK = True`, so the rest of the pipeline can be validated before Ollama is confirmed
working. **Your submitted results must be generated with `USE_MOCK = False`.**


In [10]:
def call_ollama(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 120):
    start = time.time()
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "format": "json",
                "stream": False,
                "options": {"temperature": 0.2},
            },
            timeout=timeout,
        )
        resp.raise_for_status()
        elapsed = time.time() - start
        content = resp.json()["message"]["content"]
        try:
            parsed = json.loads(content)
        except json.JSONDecodeError:
            match = re.search(r"\{.*\}", content, re.DOTALL)
            parsed = json.loads(match.group(0)) if match else None
        return parsed, content, elapsed, None
    except Exception as e:
        elapsed = time.time() - start
        return None, "", elapsed, str(e)


def mock_ollama_call(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 120):
    """Pipeline-validation stand-in only -- NOT for submitted results."""
    time.sleep(0.05)
    fake = {
        "contextualisation": "This alert reflects flow-level behaviour consistent with the "
                              "predicted category, based on the observed connection pattern.",
        "correlation_with_known_behaviour": "The connection state and packet timing broadly "
                                             "match the reference description for this category.",
        "severity": "Medium",
        "possible_impact": "Potential service degradation or unauthorised access if confirmed.",
        "immediate_response": "Rate-limit or temporarily block the source pending manual review.",
        "long_term_mitigation": "Tune detection thresholds and review access controls for the "
                                 "affected service.",
        "human_review_required": True,
        "human_review_reason": "Confidence is not high enough to act automatically without "
                                "analyst confirmation.",
    }
    return fake, json.dumps(fake), 0.05, None


def enrich_alert(record: dict):
    alert = build_cti_alert(record)
    knowledge = THREAT_KNOWLEDGE.get(alert["predicted_threat_class"], "No reference available.")
    prompt = build_prompt(alert, knowledge)
    fn = mock_ollama_call if USE_MOCK else call_ollama
    parsed, raw, elapsed, err = fn(prompt)
    return alert, parsed, raw, elapsed, err


## 7. Automated consistency / hallucination check (Part 2.3)

A cheap but genuinely useful automated check: does the LLM response mention any IP address
that isn't present in the alert itself? This won't catch every hallucination, but it catches
the clearest failure mode (inventing indicators) automatically rather than relying only on
manual review.


In [11]:
def check_hallucinated_indicators(alert: dict, response_text: str) -> list:
    ip_pattern = r"\b(?:\d{1,3}\.){3}\d{1,3}\b"
    response_ips = set(re.findall(ip_pattern, response_text))
    alert_ips = set(re.findall(ip_pattern, json.dumps(alert)))
    return list(response_ips - alert_ips)


def validate_response_schema(parsed: dict) -> list:
    """Returns a list of missing/malformed required fields."""
    required = ["contextualisation", "correlation_with_known_behaviour", "severity",
                "possible_impact", "immediate_response", "long_term_mitigation",
                "human_review_required", "human_review_reason"]
    if parsed is None:
        return required  # everything missing -- parse failure
    return [f for f in required if f not in parsed]


## 8. Run over a sample of test events

Selects a small, deliberately mixed set: a couple of confident correct predictions per class,
plus the low-confidence bruteforce false positives flagged in Part 1 -- these are the most
useful cases for checking whether the LLM's severity/human-review judgement actually reflects
uncertainty rather than just restating the predicted label.


In [12]:
import random
random.seed(42)

N_PER_CLASS = 2
classes_present = sorted(set(r["predicted_class"] for r in all_records))

sample = []
for c in classes_present:
    class_records = [r for r in all_records if r["predicted_class"] == c]
    sample.extend(random.sample(class_records, min(N_PER_CLASS, len(class_records))))

# also explicitly include any low-confidence predictions (< 0.7) -- useful edge cases
low_conf = [r for r in all_records if r["confidence"] < 0.7 and r not in sample]
sample.extend(low_conf[:5])

print(f"Running enrichment on {len(sample)} events "
      f"({len(classes_present)} classes + {len(low_conf[:5])} low-confidence edge cases).")


Running enrichment on 17 events (6 classes + 5 low-confidence edge cases).


In [13]:
results = []
for record in sample:
    alert, parsed, raw, elapsed, err = enrich_alert(record)
    missing_fields = validate_response_schema(parsed)
    invented = check_hallucinated_indicators(alert, raw) if raw else []

    results.append({
        "alert": alert,
        "true_label": record.get("true_label"),   # kept for OUR evaluation only, never sent to the LLM
        "llm_response": parsed,
        "raw_response": raw,
        "generation_time_seconds": round(elapsed, 3),
        "error": err,
        "missing_fields": missing_fields,
        "invented_indicators": invented,
    })
    status = "OK" if parsed and not missing_fields and not invented else "FLAGGED"
    print(f"[{status}] {alert['alert_id']:12s} predicted={alert['predicted_threat_class']:10s} "
          f"conf={alert['prediction_confidence']:.3f}  time={elapsed:.2f}s")

print(f"\nCompleted {len(results)} enrichments.")


[OK] CXAb1k1Dtddl7eTeXc predicted=benign     conf=1.000  time=57.84s
[OK] CXLCk73hmiEmQI3QF predicted=benign     conf=0.542  time=48.78s
[OK] CcGwVx2ldYHI6X6ee3 predicted=bruteforce conf=0.726  time=48.61s
[OK] CjSsg53Xov0zaaH8Ka predicted=bruteforce conf=0.663  time=49.76s
[OK] CIv2HH3JhtxLTz9TO7 predicted=ddos       conf=1.000  time=52.14s
[OK] CPW3cj4M9Et2C3ye8g predicted=ddos       conf=0.954  time=55.06s
[OK] CJopWu2qkhssnaNBo9 predicted=dos        conf=1.000  time=49.88s
[OK] CDapPW38vzZF1dF3q1 predicted=dos        conf=1.000  time=49.31s
[OK] CSpIgo2RqrriyHAOA1 predicted=probe      conf=1.000  time=50.11s
[OK] Con0Fw4NjG66WeXUGe predicted=probe      conf=1.000  time=51.03s
[OK] CWcc0z4qDHpDhZhW74 predicted=web        conf=1.000  time=52.01s
[OK] CrihCk3bXDTreVtTf predicted=web        conf=1.000  time=51.78s
[OK] CujAUo4QxFnee0suV1 predicted=benign     conf=0.358  time=46.86s
[OK] CJEwmt2MXPIXvRLz2l predicted=benign     conf=0.572  time=48.45s
[OK] CnCf5B3JY6K1gcpvKi predicted=br

## 9. Evaluation summary

Timing stats, how many responses were schema-complete, and how many tripped the
hallucination check -- feeds directly into the report's Discussion section (Part 2.3 /
Part 5 requirements).


In [14]:
times = [r["generation_time_seconds"] for r in results if r["error"] is None]
n_ok = sum(1 for r in results if r["llm_response"] and not r["missing_fields"])
n_flagged_hallucination = sum(1 for r in results if r["invented_indicators"])
n_errors = sum(1 for r in results if r["error"])

print(f"Total events processed:        {len(results)}")
print(f"Schema-complete responses:     {n_ok} / {len(results)}")
print(f"Flagged for invented indicators: {n_flagged_hallucination}")
print(f"Call errors:                    {n_errors}")
if times:
    print(f"Generation time -- mean: {sum(times)/len(times):.2f}s, "
          f"min: {min(times):.2f}s, max: {max(times):.2f}s")

print("\nSeverity distribution assigned by the LLM:")
from collections import Counter
severities = Counter(r["llm_response"]["severity"] for r in results if r["llm_response"])
print(dict(severities))

print("\nHuman-review-required rate:")
review_flags = Counter(r["llm_response"]["human_review_required"] for r in results if r["llm_response"])
print(dict(review_flags))


Total events processed:        17
Schema-complete responses:     17 / 17
Flagged for invented indicators: 0
Call errors:                    0
Generation time -- mean: 50.80s, min: 46.86s, max: 57.84s

Severity distribution assigned by the LLM:
{'Low': 6, 'Medium': 7, 'High': 4}

Human-review-required rate:
{True: 17}


## 10. Manual review checklist (fill in after inspecting outputs)

For your report's critical evaluation (Part 2.3), go through a handful of the results printed
above -- especially the bruteforce low-confidence cases -- and answer for each:

- Does the threat description in `contextualisation` actually match the predicted class?
- Is the assigned `severity` reasonable given the confidence and alternatives?
- Does `human_review_required` correctly come back `true` for the low-confidence/ambiguous cases?
- Any statement in the response that isn't actually supported by the alert or the reference
  knowledge (beyond what the automated IP check catches)?

This manual pass is what the exercise is actually asking for in Part 2.3 -- the automated
checks above are a starting point, not a substitute for reading the outputs.


## 11. Export results


In [15]:
with open(os.path.join(PROCESSED_DIR, "enriched_cti_alerts.json"), "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Saved {len(results)} enriched alerts to "
      f"{os.path.join(PROCESSED_DIR, 'enriched_cti_alerts.json')}")


Saved 17 enriched alerts to ../data/processed\enriched_cti_alerts.json


## 12. Pick one sample incident report

The exercise requires **one complete sample incident report submitted as a separate file**.
This selects a clear, illustrative example (prefers a genuinely enriched, non-mock response)
for that purpose -- the accompanying script formats it into a clean Word document.


In [16]:
non_mock_results = [r for r in results if not USE_MOCK]
candidates = non_mock_results if non_mock_results else results
chosen = next((r for r in candidates if r["llm_response"] and not r["missing_fields"]), candidates[0])

print("Selected alert_id:", chosen["alert"]["alert_id"])
with open(os.path.join(PROCESSED_DIR, "sample_incident_report.json"), "w") as f:
    json.dump(chosen, f, indent=2, default=str)

print(json.dumps(chosen, indent=2, default=str))


Selected alert_id: CXAb1k1Dtddl7eTeXc
{
  "alert": {
    "alert_id": "CXAb1k1Dtddl7eTeXc",
    "detection_timestamp": "",
    "predicted_threat_class": "benign",
    "prediction_confidence": 1.0,
    "alternative_predictions": [
      {
        "class": "benign",
        "confidence": 1.0
      },
      {
        "class": "bruteforce",
        "confidence": 0.0
      },
      {
        "class": "web",
        "confidence": 0.0
      }
    ],
    "affected_network_component": "O-Cloud Edge Server -- HTTP/HTTPS service",
    "network_observations": {
      "source port": 5524,
      "destination port": 80,
      "transport protocol": "tcp",
      "application service": "http",
      "flow duration (s)": 16.517246,
      "bytes sent by source": 45000.0,
      "bytes sent by destination": 1417569.0,
      "connection state": "SF",
      "missed_bytes": 0,
      "TCP flag history": "ShADadtTFf",
      "packets from source": 410,
      "src_ip_bytes": 61484,
      "packets from destination":